In [30]:
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.benchmark as benchmark
import torch._dynamo
from torchinfo import summary

In [31]:
torch._dynamo.config.cache_size_limit = 16

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = "cpu"

In [32]:
df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long).to(device)

print(X.shape)
print(y.shape)

(53940, 7)
(53940,)


In [ ]:
class DiamondNN(nn.Module):
    def __init__(self, n, num_layers=64):
        super().__init__()

        self.linears = nn.ModuleList()
        for i in range(num_layers):
            self.linears.append(nn.Linear(n + i * 2, 1))

            with torch.no_grad():
                nn.init.zeros_(self.linears[i].weight) # type: ignore
                nn.init.zeros_(self.linears[i].bias)   # type: ignore

        self.final = nn.Linear(n + 2 * num_layers, 1)
        # self.final = nn.Linear(2, 1)

        with torch.no_grad():
            self.final.weight.copy_(torch.cat([torch.zeros(n), torch.ones(2 * num_layers)]).view_as(self.final.weight))
            # nn.init.ones_(self.final.weight) 
            nn.init.zeros_(self.final.bias)

    def forward(self, x):
        for linear in self.linears:
            out = linear(x)

            pos = torch.clamp(out, max=0)
            neg = torch.clamp(out, min=0)

            x = torch.cat([x, pos, neg], dim=-1)

        x = self.final(x)
        return torch.cat([x, -x], dim=-1)

In [90]:
model = torch.compile(DiamondNN(n=X.shape[1]).to(device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [91]:
print(summary(model, input_data=X_train_tensor))

torch.set_printoptions(threshold=float('inf'))
for name, param in model.named_parameters():
    print(name)
    print(param)

epochs = 10000
best_test_loss = float('inf')

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = (test_predictions == y_test_tensor).sum().item() / y_test_tensor.size(0)

    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print()
print(f"Lowest Test Loss: {best_test_loss:.4f}")

Layer (type:depth-idx)                   Output Shape              Param #
OptimizedModule                          [43152, 2]                --
├─DiamondNN: 1-1                         [43152, 2]                --
│    └─ModuleList: 2-1                   --                        --
│    │    └─Linear: 3-1                  [43152, 1]                8
│    │    └─Linear: 3-2                  [43152, 1]                10
│    │    └─Linear: 3-3                  [43152, 1]                12
│    │    └─Linear: 3-4                  [43152, 1]                14
│    │    └─Linear: 3-5                  [43152, 1]                16
│    │    └─Linear: 3-6                  [43152, 1]                18
│    │    └─Linear: 3-7                  [43152, 1]                20
│    │    └─Linear: 3-8                  [43152, 1]                22
│    │    └─Linear: 3-9                  [43152, 1]                24
│    │    └─Linear: 3-10                 [43152, 1]                26
│    │    └─Line

KeyboardInterrupt: 

In [92]:
print()
print(f"Lowest Test Loss: {best_test_loss:.4f}")


Lowest Test Loss: 0.2870


In [93]:
for name, param in model.named_parameters():
    print(name)
    print(param)

_orig_mod.linears.0.weight
Parameter containing:
tensor([[ 0.0666, -0.0369,  0.0240, -0.0676,  0.0443, -0.1040, -0.0358]],
       requires_grad=True)
_orig_mod.linears.0.bias
Parameter containing:
tensor([-0.0256], requires_grad=True)
_orig_mod.linears.1.weight
Parameter containing:
tensor([[ 0.0042, -0.0931, -0.0095, -0.0013,  0.0090, -0.0624, -0.0198,  0.0287,
          0.1326]], requires_grad=True)
_orig_mod.linears.1.bias
Parameter containing:
tensor([-0.0783], requires_grad=True)
_orig_mod.linears.2.weight
Parameter containing:
tensor([[-0.0058, -0.0921,  0.0052,  0.0328,  0.0241, -0.0633, -0.0305,  0.0048,
          0.0435,  0.0249,  0.1338]], requires_grad=True)
_orig_mod.linears.2.bias
Parameter containing:
tensor([-0.0026], requires_grad=True)
_orig_mod.linears.3.weight
Parameter containing:
tensor([[ 0.0060, -0.0187, -0.0045, -0.0166,  0.0873, -0.0829,  0.0087,  0.0098,
          0.1083,  0.0608, -0.0387,  0.1419,  0.0082]], requires_grad=True)
_orig_mod.linears.3.bias
Parame